# 04 — Q2: 12-Month Lifetime Value

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.db import get_conn, register_csv_tables, materialize_sql_file
from src.ltv import cohort_ltv_curve, assumption_sweep, LtvScenario
from src.wise_theme import (
    apply as apply_wise_theme,
    BRIGHT_GREEN, DARK_GREEN, MID_GREEN, SOFT_GREEN, LIGHT_BG, WARM_GREY,
    diverging_cmap, sequential_cmap, style_axes, value_color, label_color_on,
)

FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
apply_wise_theme()

con = get_conn()
register_csv_tables(con)
materialize_sql_file(con, ROOT / "sql" / "01_clean_transactions.sql")
materialize_sql_file(con, ROOT / "sql" / "02_cost_lookup.sql")
materialize_sql_file(con, ROOT / "sql" / "03_profit_per_transaction.sql")
materialize_sql_file(con, ROOT / "sql" / "04_card_cohort.sql")
ts = materialize_sql_file(con, ROOT / "sql" / "05_tenure_stats.sql")
ts.round(4)

## 4. Method A — observed × extrapolated, with sensitivity bands

In [ ]:
central_ltv, central_forecast = cohort_ltv_curve(ts, LtvScenario(attrition_rate=0.10))
print(f"Central 12-month LTV (attrition=10%, anchor=T=3): GBP {central_ltv:.2f}")
central_forecast.round(4)

In [ ]:
central_ltv, central_fc  = cohort_ltv_curve(ts, LtvScenario(attrition_rate=0.10))
ltv_low,  fc_low_curve   = cohort_ltv_curve(ts, LtvScenario(attrition_rate=0.0))
ltv_high, fc_high_curve  = cohort_ltv_curve(ts, LtvScenario(attrition_rate=0.20))

fig, ax = plt.subplots(figsize=(9.0, 4.8))
ax.fill_between(central_fc["month_since_issuance"],
                fc_high_curve["cumulative_ltv_gbp"],
                fc_low_curve["cumulative_ltv_gbp"],
                alpha=0.18, color=BRIGHT_GREEN,
                label="Sensitivity (attrition 0–20%)", zorder=2)

obs_mask = central_fc["source"] == "observed"
ax.plot(central_fc.loc[obs_mask, "month_since_issuance"],
        central_fc.loc[obs_mask, "cumulative_ltv_gbp"],
        marker="o", linewidth=3.0, color=DARK_GREEN,
        markerfacecolor=DARK_GREEN, markeredgecolor=DARK_GREEN,
        label="Observed", zorder=4)
ax.plot(central_fc.loc[~obs_mask, "month_since_issuance"],
        central_fc.loc[~obs_mask, "cumulative_ltv_gbp"],
        marker="o", linewidth=3.0, color=BRIGHT_GREEN, linestyle=(0, (4, 3)),
        markerfacecolor=BRIGHT_GREEN, markeredgecolor=DARK_GREEN, markeredgewidth=1.2,
        label="Extrapolated (10% attrition)", zorder=5)

style_axes(ax, baseline=True)
ax.set_xticks(range(12))
ax.set_xlabel("Month since issuance")
ax.set_ylabel("Cumulative LTV (£)")
ax.set_title("12-month LTV per card")
ax.annotate(f"£{central_ltv:.2f}",
            xy=(11, central_fc.iloc[-1]["cumulative_ltv_gbp"]),
            xytext=(7.5, central_fc.iloc[-1]["cumulative_ltv_gbp"] + 1.6),
            fontsize=16, fontweight="bold", color=DARK_GREEN,
            bbox=dict(boxstyle="round,pad=0.4", fc=BRIGHT_GREEN, ec="none"))
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(FIG / "fig3_q2_ltv_curve.png")
plt.show()

In [ ]:
grid = assumption_sweep(ts)
grid.round(2)

In [ ]:
RESULTS = ROOT / "data" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
ts.to_csv(RESULTS / "tenure_stats.csv", index=False)
central_forecast.to_csv(RESULTS / "ltv_central_forecast.csv", index=False)
grid.to_csv(RESULTS / "ltv_sensitivity_grid.csv", index=False)
print("Persisted LTV outputs to data/results/")